In [0]:
import requests
import pandas as pd

from pyspark.sql.functions import (
    current_timestamp,
    col,
    round,
    row_number,
    avg,
    max,
    min,
    count
)
from pyspark.sql.window import Window



# 1. INGEST DATA FROM API

print("Step 1: Ingesting data from CoinGecko API...")

url = "https://api.coingecko.com/api/v3/coins/markets"

params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 50,
    "page": 1,
    "sparkline": "false"
}

response = requests.get(url, params=params, timeout=10)
response.raise_for_status()

data = response.json()

pdf = pd.DataFrame(data)

df_api = spark.createDataFrame(pdf)



# 2. BRONZE LAYER - RAW DATA


print("Step 2: Writing Bronze Layer...")

df_bronze = (
    df_api
    .withColumn("ingested_at", current_timestamp())
)

(
    df_bronze.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("crypto_bronze")
)

print("Bronze table updated successfully!")



# 3. SILVER LAYER - CLEANING


print("Step 3: Processing Silver Layer...")

df_silver = (
    spark.table("crypto_bronze")
    .select(
        col("id").alias("coin_id"),
        col("symbol"),
        col("name"),
        round(col("current_price").cast("double"), 2).alias("price_usd"),
        col("market_cap").cast("long").alias("market_cap"),
        col("total_volume").cast("long").alias("total_volume"),
        col("ingested_at")
    )
    .filter(col("coin_id").isNotNull())
    .filter(col("price_usd").isNotNull())
    .filter(col("market_cap").isNotNull())
)


# Remove exact duplicate records

df_silver = df_silver.dropDuplicates(
    ["coin_id", "ingested_at"]
)


# Save Silver

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("crypto_silver")
)

print("Silver table updated successfully!")



# 4. DATA QUALITY CHECKS


print("Step 4: Running Data Quality Checks...")

null_coins = (
    spark.table("crypto_silver")
    .filter(col("coin_id").isNull())
    .count()
)

negative_prices = (
    spark.table("crypto_silver")
    .filter(col("price_usd") < 0)
    .count()
)

negative_market_caps = (
    spark.table("crypto_silver")
    .filter(col("market_cap") < 0)
    .count()
)

print("Null coin IDs:", null_coins)
print("Negative prices:", negative_prices)
print("Negative market caps:", negative_market_caps)

if null_coins > 0 or negative_prices > 0 or negative_market_caps > 0:
    raise Exception("Data Quality Check Failed!")

print("Data Quality Checks Passed!")



# 5. GOLD LAYER - LATEST SNAPSHOT


print("Step 5: Creating Gold Layer...")

df_silver = spark.table("crypto_silver")

window = Window.partitionBy("coin_id").orderBy(
    col("ingested_at").desc()
)

df_latest = (
    df_silver
    .withColumn("row_num", row_number().over(window))
    .filter(col("row_num") == 1)
    .drop("row_num")
)


(
    df_latest.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("crypto_gold_latest")
)

print("Gold latest snapshot created!")



# 6. GOLD LAYER - MARKET ANALYTICS


print("Step 6: Creating Gold Analytics...")

df_gold = (
    df_silver
    .groupBy("coin_id", "symbol", "name")
    .agg(
        round(avg("price_usd"), 2).alias("avg_price_usd"),
        round(max("price_usd"), 2).alias("max_price_usd"),
        round(min("price_usd"), 2).alias("min_price_usd"),
        max("market_cap").alias("max_market_cap"),
        max("total_volume").alias("max_total_volume"),
        count("*").alias("number_of_snapshots")
    )
)


(
    df_gold.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("crypto_gold_analytics")
)

print("Gold analytics table created!")



# 7. VERIFY RESULTS


print("Pipeline completed successfully!")

display(
    spark.sql("""
        SELECT *
        FROM crypto_gold_latest
        ORDER BY market_cap DESC
        LIMIT 10
    """)
)

display(
    spark.sql("""
        SELECT *
        FROM crypto_gold_analytics
        ORDER BY max_market_cap DESC
        LIMIT 10
    """)
)

Step 1: Ingesting data from CoinGecko API...
Step 2: Writing Bronze Layer...
Bronze table updated successfully!
Step 3: Processing Silver Layer...
Silver table updated successfully!
Step 4: Running Data Quality Checks...
Null coin IDs: 0
Negative prices: 0
Negative market caps: 0
Data Quality Checks Passed!
Step 5: Creating Gold Layer...
Gold latest snapshot created!
Step 6: Creating Gold Analytics...
Gold analytics table created!
Pipeline completed successfully!


coin_id,symbol,name,price_usd,market_cap,total_volume,ingested_at
bitcoin,btc,Bitcoin,77102.0,1548494208935,15749441428,2026-09-13T16:00:09.625Z
ethereum,eth,Ethereum,2489.8,303882386200,8702566964,2026-09-13T16:00:09.625Z
tether,usdt,Tether,1.0,183475631772,31889014229,2026-09-13T16:00:09.625Z
binancecoin,bnb,BNB,718.62,95688800444,729645781,2026-09-13T16:00:09.625Z
ripple,xrp,XRP,1.34,84490620375,939331140,2026-09-13T16:00:09.625Z
usd-coin,usdc,USDC,1.0,74222370716,6463781352,2026-09-13T16:00:09.625Z
solana,sol,Solana,100.23,58809159358,1856686698,2026-09-13T16:00:09.625Z
tron,trx,TRON,0.34,32389606802,319670713,2026-09-13T16:00:09.625Z
figure-heloc,figr_heloc,Figure Heloc,1.0,22551817529,50043,2026-09-13T16:00:09.625Z
zcash,zec,Zcash,1087.55,18413673023,1186802239,2026-09-13T16:00:09.625Z


coin_id,symbol,name,avg_price_usd,max_price_usd,min_price_usd,max_market_cap,max_total_volume,number_of_snapshots
bitcoin,btc,Bitcoin,78527.07,79392.0,77102.0,1594356735661,35141729269,15
ethereum,eth,Ethereum,2496.4,2545.62,2465.35,310641378821,24277994213,15
tether,usdt,Tether,1.0,1.0,1.0,183488296223,70206988977,15
binancecoin,bnb,BNB,740.01,755.97,714.07,100666555595,1233461023,15
ripple,xrp,XRP,1.4,1.44,1.34,90187271396,2762214431,15
usd-coin,usdc,USDC,1.0,1.0,1.0,74427772301,19351455830,15
solana,sol,Solana,103.08,104.39,99.84,61193256374,4341690199,15
tron,trx,TRON,0.34,0.34,0.33,32389606802,491971761,15
figure-heloc,figr_heloc,Figure Heloc,1.04,1.06,1.0,23516084575,239043263,15
zcash,zec,Zcash,1186.68,1270.19,1087.55,21498409111,1945266444,15
